In [ ]:
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
import torch
from PIL import Image
import requests
import matplotlib.pyplot as plt
import cv2


processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

prompts = ["cutlery", "pancakes", "blueberries", "orange juice"]

# url = "https://unsplash.com/photos/8Nc_oQsc2qQ/download?ixid=MnwxMjA3fDB8MXxhbGx8fHx8fHx8fHwxNjcxMjAwNzI0&force=true&w=640"
# image = Image.open(requests.get(url, stream=True).raw)
image = cv2.imread("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/datasets/our_office_static/image_1771496652884.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# prompt = Image.open(requests.get(url, stream=True).raw)
inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
# predict
with torch.no_grad():
  outputs = model(**inputs)
preds = outputs.logits.unsqueeze(1)

_, ax = plt.subplots(1, len(prompts) + 1, figsize=(3*(len(prompts) + 1), 4))
[a.axis('off') for a in ax.flatten()]
ax[0].imshow(image)
[ax[i+1].imshow(torch.sigmoid(preds[i][0])) for i in range(len(prompts))];
[ax[i+1].text(0, -15, prompt) for i, prompt in enumerate(prompts)];

In [ ]:
prompts = ["segment humans "]

# url = "https://unsplash.com/photos/8Nc_oQsc2qQ/download?ixid=MnwxMjA3fDB8MXxhbGx8fHx8fHx8fHwxNjcxMjAwNzI0&force=true&w=640"
# image = Image.open(requests.get(url, stream=True).raw)
image = cv2.imread("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/datasets/our_office_static/image_1771496652884.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# prompt = Image.open(requests.get(url, stream=True).raw)
inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
# predict
with torch.no_grad():
  outputs = model(**inputs)
preds = outputs.logits.unsqueeze(1)

_, ax = plt.subplots(1, len(prompts) + 1, figsize=(3*(len(prompts) + 1), 4))
[a.axis('off') for a in ax.flatten()]
ax[0].imshow(image)
[ax[i+1].imshow(torch.sigmoid(preds[i][0])) for i in range(len(prompts))];
[ax[i+1].text(0, -15, prompt) for i, prompt in enumerate(prompts)];

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import OxfordIIITPet
import tqdm

# ======================
# Configuration
# ======================
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Transforms
# ======================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================
# Dataset
# ======================
train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=transform
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# ======================
# Model
# ======================
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37)
model = model.to(DEVICE)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ======================
# Training Loop
# ======================
for epoch in tqdm.tqdm(range(EPOCHS)):

    # ---- TRAINING ----
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    # ---- TESTING ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"- Loss: {epoch_loss:.4f} "
          f"- Test Accuracy: {accuracy:.2f}%")

print("Training Complete!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
import timm
import tqdm

# ======================
# Configuration
# ======================
BATCH_SIZE = 32
LR = 3e-4
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Transforms
# ======================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================
# Dataset
# ======================
train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=transform
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# ======================
# Model (ViT-base from timm)
# ======================
model = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    num_classes=37
)

model = model.to(DEVICE)

# ======================
# Loss & Optimizer
# ======================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# ======================
# Training Loop
# ======================
for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    running_loss = 0.0

    for images, labels in tqdm.tqdm(train_loader, leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    # ---- EVALUATION ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"- Loss: {epoch_loss:.4f} "
          f"- Test Accuracy: {accuracy:.2f}%")

print("Training Complete!")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
import timm
import tqdm
import math

# ======================
# Configuration
# ======================
BATCH_SIZE = 32
LR = 5e-5
EPOCHS = 10
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Transforms
# ======================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======================
# Dataset
# ======================
train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=transform
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# ======================
# Model
# ======================
model = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    num_classes=37
)
model = model.to(DEVICE)

# ======================
# Optimizer
# ======================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss()

# ======================
# Scheduler (Warmup + Cosine)
# ======================
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(WARMUP_RATIO * total_steps)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ======================
# Training Loop
# ======================
global_step = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    running_loss = 0.0

    for images, labels in tqdm.tqdm(train_loader, leave=False):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        global_step += 1

    epoch_loss = running_loss / len(train_loader)

    # ---- EVALUATION ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            ic(outputs.shape)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"- Loss: {epoch_loss:.4f} "
          f"- Test Accuracy: {accuracy:.2f}%")

print("Training Complete!")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
import timm
from tqdm.notebook import tqdm
import math

def set_seed(seed: int):
    # random.seed(seed)
    # np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:

BATCH_SIZE = 16
LR = 5e-6
EPOCHS = 20
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=transform
)

test_dataset = OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

from vit_tiny import vit_tiny_classifier
model = vit_tiny_classifier(n_classes=37)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss()


total_steps = EPOCHS * len(train_loader)
warmup_steps = int(WARMUP_RATIO * total_steps)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

global_step = 0

for epoch in tqdm(range(EPOCHS)):

    # ---- TRAIN ----
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        global_step += 1

    epoch_loss = running_loss / len(train_loader)

    # ---- EVALUATION ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            # ic(outputs.shape)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"- Loss: {epoch_loss:.4f} "
          f"- Test Accuracy: {accuracy:.2f}%")

print("Training Complete!")

 70%|███████   | 161/230 [00:03<00:01, 42.72it/s]